# 📥 POLYDIM V4.2 - AGENTE RECEPTOR (RECEIVER)
Versión FFT Pura: Reconstrucción Algebraicamente Exacta y Timestamps.


In [ ]:
!pip install -q torch numpy
import torch
import numpy as np
import time
import os
import torch.nn.functional as F
from google.colab import drive
import math

drive.mount('/content/drive')
BUS_DIR = "/content/drive/MyDrive/POLYDIM_BUS"
os.makedirs(BUS_DIR, exist_ok=True)
print(f"✅ Escuchando bus en: {BUS_DIR}")



In [ ]:
# MOTOR ISOMÉTRICO V4.2
class IsometricRotation(torch.nn.Module):
    def __init__(self, D: int, seed=42):
        super().__init__()
        self.D = D
        phase_scale = min(0.01, 1.0 / math.sqrt(D))
        gen = torch.Generator().manual_seed(seed)
        self.phase = torch.nn.Parameter(torch.randn(D, generator=gen) * phase_scale)
        self.phase.requires_grad = False
        
    def inverse(self, x: torch.Tensor) -> torch.Tensor:
        # Inversa exacta de la FFT con fase
        x = torch.fft.fft(x, dim=-1)
        x = x * torch.exp(-1j * self.phase).unsqueeze(0)
        x = torch.fft.ifft(x, dim=-1)
        return x.real



In [ ]:
# BUCLE DE RECEPCIÓN Y EVALUACIÓN
processed = set()
print("🚀 ESCUCHANDO PMTP V4.2 (FFT PURA)...")
print("="*75)
print(f"{'CHUNK':<12} | {'D':<6} | {'DECODE(ms)':<12} | {'MSE':<15} | {'STATUS'}")
print("="*75)

while True:
    try:
        files = sorted([f for f in os.listdir(BUS_DIR) if f.endswith(".pt")])
        for f in files:
            if f in processed: continue
            
            chunk_id = int(f.split("_")[1].split(".")[0])
            file_path = os.path.join(BUS_DIR, f)
            
            package = torch.load(file_path, weights_only=True)
            ecp_encoded = package['tensor']
            seed_rot = package['seed_rot']
            original_val = package['original_for_validation_only']
            
            d_detectado = ecp_encoded.shape[-1]
            
            t0 = time.time()
            rotor_rx = IsometricRotation(d_detectado, seed=seed_rot)
            ecp_decoded = rotor_rx.inverse(ecp_encoded)
            t_decode = (time.time() - t0) * 1000
            
            mse = F.mse_loss(original_val, ecp_decoded).item()
            status = "✅ OK" if mse < 1e-10 else "⚠️ DEGRADADO"
            
            print(f"chunk_{chunk_id:04d} | D={d_detectado:<4} | {t_decode:<12.2f} | {mse:<15.2e} | {status}")
            processed.add(f)
            
        time.sleep(2)
    except KeyboardInterrupt:
        break

